# Hyperparameter search on the BBBP data set

This is a more advanced example of how to use MolPipeline in combination with Sklearn to make molecular machine learning experiments. In this notebook we show how hyperparameter optimization can be performed using MolPipeline and Sklearn's [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV). We will train multiple Sklearn estimators (like RandomForest) with Morgan fingerprints and scaffold splits and select the best performing model.

We selected the BBBP data sets from [MoleculeNet](https://moleculenet.org/) because it is a widely used data sets for benchmarking molecular property prediction methods and it contains some problematic molecules that are erroneous which highlights the error handling capabilities of MolPipeline. See [the blog post](http://practicalcheminformatics.blogspot.com/2023/08/we-need-better-benchmarks-for-machine.html) by Patrick Walters for details on the erroneous molecules.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier

from molpipeline import ErrorFilter, FilterReinserter, Pipeline, PostPredictionWrapper
from molpipeline.abstract_pipeline_elements.core import MolToMolPipelineElement
from molpipeline.any2mol import AutoToMol
from molpipeline.estimators import MurckoScaffoldClustering
from molpipeline.mol2any import MolToMorganFP
from molpipeline.mol2mol import (
    ElementFilter,
    MetalDisconnector,
    SaltRemover,
    StereoRemover,
    TautomerCanonicalizer,
    Uncharger,
)

In [ ]:
N_JOBS = 16
RANDOM_STATE = 123456

Let's start by getting the data set

In [ ]:
df = pd.read_csv(
    "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
    usecols=lambda col: col != "num",
)
df

Make a function to setup some pipeline elements for standardization that we will use later

In [ ]:
def make_standardization_elements() -> list[tuple[str, MolToMolPipelineElement]]:
    """Make a new list of pipeline elements for molecular standardization.

    Returns
    -------
    list[tuple[str, MolToMolPipelineElement]]
        List of tuples with the name and the pipeline element.

    """
    return [
        (
            "element_filter",
            ElementFilter(
                allowed_element_numbers=[
                    1,
                    3,
                    5,
                    6,
                    7,
                    8,
                    9,
                    11,
                    12,
                    14,
                    15,
                    16,
                    17,
                    19,
                    20,
                    34,
                    35,
                    53,
                ],
            ),
        ),
        ("metal_disconnector", MetalDisconnector()),
        ("salt_remover", SaltRemover()),
        ("uncharge", Uncharger()),
        ("canonical_tautomer", TautomerCanonicalizer()),
        ("stereo_remover", StereoRemover()),
    ]

### Make a clustering pipeline

We first setup a pipeline that will do the scaffold clustering for us using [Murcko scaffolds](https://www.rdkit.org/docs/GettingStartedInPython.html#murcko-decomposition).

In this example we also use Molpipelines error handling to detect and filter out all invalid molecules before clustering. After clustering we will re-add fill-values(np.nan in this case) at the corresponding positions of the invalid molecules. This keeps the resulting list of cluster labels (also called 'groups') in the same shape as the input molecule list. 

In [ ]:
# create an error filter and reinserter.
error_filter = ErrorFilter()
# the Reinserter is wrapped into a PostPredictionWrapper because we execute
# the re-insertion step at the end of the pipeline, even after the clustering.
error_reinserter = PostPredictionWrapper(
    FilterReinserter.from_error_filter(error_filter, np.nan),
)

# set up the clustering pipeline
pipeline_clustering = Pipeline(
    [
        ("auto2mol", AutoToMol()),
        *make_standardization_elements(),
        # filter out invalid samples before clustering
        ("error_filter", error_filter),
        (
            "scaffoldClustering",
            MurckoScaffoldClustering(linear_molecules_strategy="own_cluster"),
        ),
        # re-add fill values to invalid samples to keep the input shape
        ("error_reinserter", error_reinserter),
    ],
    n_jobs=N_JOBS,
)

# cluster the smiles by their murcko scaffold
df["groups"] = pipeline_clustering.fit_predict(df["smiles"])

# the 'groups' are the cluster labels
df["groups"]

We can have a look at the clustering results by plotting the distribution of cluster sizes.

In [ ]:
values, counts = np.unique(df["groups"], return_counts=True)
print("Number of cluster:", len(values))
plt.bar(range(len(counts)), sorted(counts))
plt.xlabel("Cluster rank by size")
plt.ylabel("Number of molecules in cluster")
plt.yscale("log")
_ = plt.title("Sorted distribution of cluster sizes")

### Analyzing failed molecules

For the BBBP data sets some molecules fail the processing by the clustering pipeline.
Analysing the molecules that failed is an optional step but can give interesting insights.

Since we replaced all failed molecules with np.nan we can count them with numpy's isnan function 

In [ ]:
df["groups"].isna().sum()

Using the nan values in the `df["groups"]` array we can map back to the original molecules in the BBBP data set.

In [ ]:
df.loc[df["groups"].isna(), ["smiles"]]

These molecules contain uncharged tetravalent nitrogen atoms which is not correct. Therefore, these SMILES represent invalid structures and RDKit can't read them. See [Pat Walters great blog post](http://practicalcheminformatics.blogspot.com/2023/08/we-need-better-benchmarks-for-machine.html) for more details. Molpipeline can automatically handle these cases through it's error handling. Without the error handling the execution of the pipeline would fail and we would have to make manually interventions.   

### Group clusters into train/valid/test split

We now will group the computed clusters into train/valid/test splits that we can use with Sklearn's GridSearchCV. The split ratio we choose is 80/10/10.

In [ ]:
TEST_SET_SIZE = 0.1  # 10%
VALID_SET_SIZE = 0.1  # 10%

# create a mask to remove invalid samples from all data
df_has_grouping = df.query("groups.notna()")

# we first do a 90/10 split to create the hold-out test set
grouper_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=RANDOM_STATE)

# test_indices are the indices of the hold-out test set.
tmp_indices, test_indices = next(
    grouper_test.split(
        df_has_grouping["smiles"],
        df_has_grouping["p_np"],
        groups=df_has_grouping["groups"],
    ),
)

# calculate what 10% of the full data set is on the 90% chunk
propotion_valid_set = df["smiles"].shape[0] * VALID_SET_SIZE / len(tmp_indices)

# then, we split the 90% of the data again to create a train and a valid set.
grouper_valid = GroupShuffleSplit(
    n_splits=1,
    test_size=propotion_valid_set,
    random_state=RANDOM_STATE,
)

# the 10% as hold-out test set
df_test = df_has_grouping.iloc[test_indices, :]

# the 90% containing both train and valid set (will be used later with the grouper_valid)
df_train_and_valid = df_has_grouping.iloc[tmp_indices, :]
df_train_and_valid

### Hyperparameter search and model selection

Here we will use Sklearn's GridSearchCV with Molpipeline to determine the best hyperparameter configuration on BBBP using the train and valid data sets. Afterwards we can determine the best models' performance on the hold-out test set.

Before starting the actual hyperparameter search we pre-compute the standardized molecules. While this pre-computation is not strictly necessary this gives a speed up of about 5-6x for the whole hyperparameter search. The slowest step in the standardization is finding a canonical tautomer. By pre-computation the canonical tautomer is determined once and not for each hyperparameter combination.

In [ ]:
%%time

# Let's pre-compute the molecular standardization
pipeline_standardization = Pipeline(
    [("auto2mol", AutoToMol()), *make_standardization_elements()],
    n_jobs=N_JOBS,
)

stand_mols = pipeline_standardization.transform(df_train_and_valid["smiles"])

Let's do the hyperparameter search

In [ ]:
%%time

# setup the estimators and their hyperparameters we want to consider in the grid search
estimator_dict = {
    "RF": {
        "model": RandomForestClassifier(),
        "hyperparameters": {
            "n_estimators": [200, 1000],
            "max_depth": [8, 32, 128, None],
            "min_samples_split": [2, 5, 7],
            "min_samples_leaf": [1, 3, 5],
            "random_state": [RANDOM_STATE],
        },
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "hyperparameters": {
            "n_neighbors": [3, 5],
            "metric": ["jaccard"],
            "weights": ["uniform", "distance"],
        },
    },
    "Logit": {
        "model": LogisticRegression(),
        "hyperparameters": {
            "C": [0.001, 0.1, 1, 10, 100, 1000],
            "max_iter": [1000],
            "random_state": [RANDOM_STATE],
        },
    },
}

# Molpipline lets you also natively include hyperparameters of the descriptor
morgan_grid_spaces = {"morgan__n_bits": [1024, 2048], "morgan__radius": [1, 2]}

# we loop over all estimators and do a grid search separately
cv_dict = {}
for model_name, model_dict in estimator_dict.items():
    pipeline_estimator = Pipeline(
        [
            ("auto2mol", AutoToMol()),
            ("morgan", MolToMorganFP(return_as="dense")),
            ("estimator", model_dict["model"]),
        ],
        n_jobs=N_JOBS,
    )

    pipeline_param_grid = {
        f"estimator__{k}": v for k, v in model_dict["hyperparameters"].items()
    }
    pipeline_param_grid.update(morgan_grid_spaces)

    cv = GridSearchCV(
        estimator=pipeline_estimator,
        param_grid=pipeline_param_grid,
        cv=grouper_valid,
        scoring="roc_auc",
        n_jobs=N_JOBS,
    )

    cv.fit(
        X=stand_mols,
        y=df_train_and_valid["p_np"],
        groups=df_train_and_valid["groups"],
    )

    cv_dict[model_name] = cv

Let's have a look at the best estimators selected with the grid search

In [ ]:
best_estimators = [cv.best_estimator_ for cv in cv_dict.values()]
best_estimators

The three estimators achieved the following ROC AUCs on the validation set

In [ ]:
best_valid_scores = [cv.best_score_ for cv in cv_dict.values()]
best_valid_scores

### Determine performance on the hold-out test set

After selecting the best predictors on the valid set we want to see an estimate of their performance on the hold-out test set  

In [ ]:
# pre-compute standardized molecules
pipeline_standardization = Pipeline(
    [("auto2mol", AutoToMol()), *make_standardization_elements()],
    n_jobs=N_JOBS,
)
test_smiles_standardized = pipeline_standardization.transform(df_test["smiles"])

In [ ]:
test_roc_aucs = {}
for model_name, cv in cv_dict.items():
    best_estimator = cv.best_estimator_
    test_predictions = best_estimator.predict_proba(test_smiles_standardized)
    test_roc_aucs[model_name] = roc_auc_score(df_test["p_np"], test_predictions[:, 1])
    print(
        model_name,
        f"valid_roc_auc: {cv.best_score_:.2f}",
        f"test_roc_auc: {test_roc_aucs[model_name]:.2f}",
    )

As shown above the best model on the validation and test set is RandomForestClassifier. However, the other two simpler models are also good predictors with ROC AUCs of around 0.90. 

### Plot the performances on the validation set

Let's extract the result on the validation set from all hyperparameter combinations we tried in a DataFrame for plotting.

In [ ]:
dict_data = {"mean_test_score": [], "estimator": []}
for model_name, cv in cv_dict.items():
    dict_data["mean_test_score"].extend(cv.cv_results_["mean_test_score"])
    dict_data["estimator"].extend([model_name] * len(cv.cv_results_["mean_test_score"]))

rename_dict = {
    "RF": "RandomForest",
    "KNN": "K-NearestNeigbor",
    "Logit": "LogisticRegression",
}

df_cv = pd.DataFrame(dict_data)
df_cv["estimator"] = df_cv["estimator"].replace(rename_dict)
df_cv

Now, let's plot the performances of all tried hyperparameter configurations of the three models. The red X's are the performances of the best estimators on the hold-out test set.

In [ ]:
plt.figure(figsize=(8, 9))

# plot the validation set performance
ax = sns.boxplot(
    data=df_cv,
    x="estimator",
    y="mean_test_score",
    hue="estimator",
    boxprops={"alpha": 0.4},
    showmeans=False,
)
sns.stripplot(
    data=df_cv,
    x="estimator",
    y="mean_test_score",
    hue="estimator",
    ax=ax,
)

# plot the test set performance of the best models
for model_name, metric_value in test_roc_aucs.items():
    plt.scatter(
        rename_dict[model_name],
        metric_value,
        marker="x",
        color="red",
        zorder=10,
    )

plt.title("Performance on the validation and test set", fontsize=16)
plt.ylabel("ROC AUC", fontsize=13)
plt.xlabel("Estimators", fontsize=13)
plt.xticks(fontsize=12)
plt.tight_layout()

The results illustrate the ROC AUC distribution of the models' hyperparameter configurations during the optimization on the validation set and after the parameter selection on the test set. The models vary in their performance. Overall, the prediction performance on the BBBP data set is very good, even with KNN and logistic regression.